<a href="https://colab.research.google.com/github/venkata18167/CSA6102-DIGITAL-FORENSICS-AND-CYBERCRIME-INVESTIGATION/blob/main/29_Windows_Event_Log_Brute_Force_Login_Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
 import pandas as pd
from google.colab import files

print("Upload Windows Event Log CSV File")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
print("\n========== Uploaded Dataset ==========")
print(df)
FAILED_EVENT = 4625
SUCCESS_EVENT = 4624
THRESHOLD = 5
df["EventID"] = df["EventID"].astype(int)
failed_df = df[df["EventID"] == FAILED_EVENT]
success_df = df[df["EventID"] == SUCCESS_EVENT]
failed_counts = failed_df.groupby("IPAddress").size().reset_index(name="FailedAttempts")

alerts = []
for _, row in failed_counts.iterrows():
    ip = row["IPAddress"]
    attempts = row["FailedAttempts"]

    if attempts >= THRESHOLD:

        success = success_df[success_df["IPAddress"] == ip]

        if not success.empty:
            alerts.append({
                "IP Address": ip,
                "Failed Attempts": attempts,
                "Successful Login": "YES",
                "Username": success.iloc[0]["Username"],
                "Risk Level": "HIGH"
            })
        else:
            alerts.append({
                "IP Address": ip,
                "Failed Attempts": attempts,
                "Successful Login": "NO",
                "Username": "-",
                "Risk Level": "MEDIUM"
            })

print("\n")
print("=" * 70)
print("        WINDOWS EVENT LOG BRUTE-FORCE LOGIN DETECTOR")
print("=" * 70)

if alerts:
    result = pd.DataFrame(alerts)
    print(result.to_string(index=False))
else:
    print("No brute-force login attempts detected.")

print("=" * 70)

Upload Windows Event Log CSV File


Saving windows_event_log_sample.csv to windows_event_log_sample.csv

========== Uploaded Dataset ==========
     Time  EventID Username     IPAddress
0   09:00     4625    Admin  192.168.1.10
1   09:01     4625    Admin  192.168.1.10
2   09:02     4625    Admin  192.168.1.10
3   09:03     4625    Admin  192.168.1.10
4   09:04     4625    Admin  192.168.1.10
5   09:05     4624    Admin  192.168.1.10
6   09:10     4625     John    10.10.10.5
7   09:11     4625     John    10.10.10.5
8   09:12     4625     John    10.10.10.5
9   09:13     4625     John    10.10.10.5
10  09:14     4625     John    10.10.10.5
11  09:20     4625    Guest   172.16.1.15


        WINDOWS EVENT LOG BRUTE-FORCE LOGIN DETECTOR
  IP Address  Failed Attempts Successful Login Username Risk Level
  10.10.10.5                5               NO        -     MEDIUM
192.168.1.10                5              YES    Admin       HIGH
